In [1]:
import numpy as np
import SWMM_ENV
import PPO as PPO
import tensorflow as tf

D:\anaconda3\envs\py38\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
env_params={
        'orf':'chaohu',
        'advance_seconds':300
    }
env=SWMM_ENV.SWMM_ENV(env_params)

raindata = np.load('training_raindata.npy').tolist()

agent_params={
    'state_dim':len(env.config['states']),
    'action_dim':int(2**len(env.config['action_assets'])),
    'actornet_layer':[30,30,30],
    'criticnet_layer':[30,30,30],
    
    'bound_low':0,
    'bound_high':1,
    
    'clip_ratio':0.01,
    'target_kl':0.03,
    'lam':0.01,
    
    'policy_learning_rate':0.001,
    'value_learning_rate':0.001,
    'train_policy_iterations':20,
    'train_value_iterations':20,
    
    'num_rain':2,
    
    'training_step':100,
    'gamma':0.3,
    'epsilon':1,
    'ep_min':1e-50,
    'ep_decay':0.1
}
model = PPO.PPO(agent_params,env)

In [14]:
model.actor.trainable_variables[0].value

<bound method BaseResourceVariable.value of <tf.Variable 'dense/kernel:0' shape=(18, 30) dtype=float32, numpy=
array([[ 2.35506892e-03, -1.29022032e-01, -2.71741986e-01,
         2.39380509e-01,  3.22652757e-02, -2.65475482e-01,
        -5.32600284e-03, -2.19983533e-01, -2.67558813e-01,
         1.36567920e-01,  3.73101234e-03,  3.10462207e-01,
         3.53120059e-01, -3.50309849e-01,  2.50994831e-01,
         3.31867069e-01, -2.11079508e-01, -4.35734391e-02,
         2.97409207e-01, -1.90318733e-01,  6.86073899e-02,
         3.30794603e-01,  5.05784154e-02,  1.54158086e-01,
        -9.89204645e-03, -1.26498029e-01, -2.64157057e-01,
         1.62714750e-01,  2.23701894e-02,  3.33878726e-01],
       [ 3.07426125e-01,  1.06826335e-01,  1.18805379e-01,
         8.19371939e-02,  7.53779411e-02,  3.94640267e-02,
        -9.16854143e-02,  2.47343093e-01,  2.90640801e-01,
        -6.26455545e-02, -1.01637036e-01,  1.94847077e-01,
         3.16176802e-01,  1.79308027e-01, -6.53766692e-02,
   

In [15]:
model.load_model()

In [16]:
model.actor.trainable_variables[0].value

<bound method BaseResourceVariable.value of <tf.Variable 'dense/kernel:0' shape=(18, 30) dtype=float32, numpy=
array([[ 0.20163491, -0.07866315,  0.29577416, -0.1044715 ,  0.35138765,
        -0.15758765,  0.04525153, -0.16069844, -0.32826808,  0.27452403,
         0.1802594 ,  0.05682496,  0.07598877, -0.25085923,  0.0816257 ,
        -0.04659691, -0.10361996, -0.27849895, -0.13412672, -0.0831475 ,
        -0.17116883,  0.21280971, -0.08279707, -0.05723611, -0.11066217,
        -0.2862621 ,  0.0754907 ,  0.217702  ,  0.34606084,  0.13613844],
       [ 0.26283774, -0.02471485, -0.2728297 , -0.07431047, -0.16538677,
         0.27402395,  0.11742242,  0.00214202,  0.29072973,  0.13794708,
         0.20628138, -0.0964919 , -0.22671409,  0.12572542,  0.05246551,
         0.17464352, -0.12317264, -0.05383746,  0.08999707, -0.05564924,
        -0.30003178, -0.20002489, -0.18787655,  0.15523037,  0.05352695,
        -0.3340808 , -0.1169422 ,  0.1680143 ,  0.34881172,  0.28424853],
       [-0.

# DDQN BC code

In [ ]:
if self.params['ddqn']:
            #DDQN
            #eval net
            A_prev = self.s
            for i in np.arange(self.params['evalnet_layer_A']):
                A_prev=layers.Dense(self.params['evalnet_A'][i]['num'], activation='relu', name='evalnet_A'+str(i))(A_prev)
            V_prev = self.s
            for i in np.arange(self.params['evalnet_layer_V']):
                V_prev=layers.Dense(self.params['evalnet_V'][i]['num'], activation='relu', name='evalnet_V'+str(i))(V_prev)
            A_prev_avg = A_prev-tf.reduce_mean(A_prev)
            self.eval_out = layers.Add()([V_prev,A_prev_avg])
            
            #target net
            An_prev = self.s_next
            for i in np.arange(self.params['targetnet_layer_A']):
                An_prev=layers.Dense(self.params['targetnet_A'][i]['num'], activation='relu', name='targetnet_A'+str(i))(An_prev)
            Vn_prev = self.s
            for i in np.arange(self.params['targetnet_layer_V']):
                Vn_prev=layers.Dense(self.params['targetnet_V'][i]['num'], activation='relu', name='targetnet_V'+str(i))(Vn_prev)
            An_prev_avg = An_prev-tf.reduce_mean(An_prev)
            self.target_out = layers.Add()([Vn_prev,An_prev_avg])
        else:
            

# DQN code

In [6]:
class DQN:
    
    def __init__(self,params,env):
        tf.compat.v1.disable_eager_execution()
        self.params=params
        self.memory_buffer = deque(maxlen=2000)
        self.action_table=pd.read_excel('./action_table_of_DQN.xlsx').values[:,1:]
        print('table shape: ',self.action_table.shape)
        
        self.model=_build_net()
        self.target_model=_build_net()
        self.update_target_model()
        
    def _build_net(self):
        self.s = layers.Input(shape=self.params['state_dim'],name='s_input')
        self.s_next = layers.Input(shape=self.params['state_dim'],name='s_next_input')
        self.q_target = layers.Input(shape=self.params['action_dim'],name='qt_input')
        
        #DQN
        #eval net
        V_prev = self.s
        for i in np.arange(self.params['evalnet_layer_V']):
            V_prev=layers.Dense(self.params['evalnet_V'][i]['num'], activation='relu', name='evalnet_V'+str(i))(V_prev)
        
        model=models.Model(inputs=[self.s,self.s_next,self.q_target],outputs=self.eval_out)
        return model
    
    def choose_action(self,state,train_log):
        #input state, output action
        if train_log:
            #epsilon greedy
            pa = np.random.uniform()
            if pa < self.params['epsilon']:
                action_value = self.model.predict([state])
                action = np.argmax(action_value)
            else:
                action = np.random.randit(0,self.params['action_dim'])
        else:
            action_value = self.model.predict([state])
            action = np.argmax(action_value)
        return action

    
    def remember(self, state, action, reward, next_state, done):
        item = (state, action, reward, next_state, done)
        self.memory_buffer.append(item)

    def process_batch(self, batch):
         # 从经验池中随机采样一个batch
        data = random.sample(self.memory_buffer, batch)
        # 生成Q_target。
        states = np.array([d[0] for d in data])
        next_states = np.array([d[3] for d in data])

        y = self.model.predict(states)
        q = self.target_model.predict(next_states)

        for i, (_, action, reward, _, done) in enumerate(data):
            target = reward
            if not done:
                target += self.gamma * np.amax(q[i])
            y[i][action] = target
        return states, y
    
    
    def train(self,total_step):
        #sampling and upgrading
        for j in range(self.params['training_step']):
            total_step=0
            for i in range(self.params['num_rain']):
                print('training step:',j,' sampling num:',i)
                #Sampling: each rainfall represent one round of sampling
                s = self.env.reset(self.RainData[i])
                done, batch = False, 0
                while not done:
                    a = self.choose_action(s,True)
                    action = self.action_table[a,:].tolist()
                    snext,reward,done = self.env.step(action)
                    self.remember(s, a, reward, snext, done)
                    s = snext
                    batch+=1
                
                #Upgrading: each rainfall for one round of upgrading
                X, y = self.process_batch(batch)
                loss = self.model.train_on_batch(X, y)
                count += 1
                # 减小egreedy的epsilon参数。
                if self.epsilon >= self.epsilon_min:
                    self.epsilon *= self.epsilon_decay
                # 固定次数更新target_model
                if count != 0 and count % 20 == 0:
                    self.target_model.set_weights(self.model.get_weights())
                if i % 5 == 0:
                    history['episode'].append(i)
                    history['Episode_reward'].append(reward_sum)
                    history['Loss'].append(loss)

                    print('Episode: {} | Episode reward: {} | loss: {:.3f} | e:{:.2f}'.format(i, reward_sum, loss, self.epsilon))

            self.model.save_weights('./model/dqn.h5')
        return history
    
    def play(self):
        observation = self.env.reset()

        count = 0
        reward_sum = 0
        random_episodes = 0

        while random_episodes < 10:
            self.env.render()

            x = observation.reshape(-1, 4)
            q_values = self.model.predict(x)[0]
            action = np.argmax(q_values)
            observation, reward, done, _ = self.env.step(action)

            count += 1
            reward_sum += reward

            if done:
                print("Reward for this episode was: {}, turns was: {}".format(reward_sum, count))
                random_episodes += 1
                reward_sum = 0
                count = 0
                observation = self.env.reset()

        self.env.close()

In [4]:
import SWMM_ENV

In [5]:
params={
    'state_dim':5,
    'action_dim':8,
    'evalnet_layer_A':3,
    'evalnet_A':[{'num':30},{'num':30},{'num':30}],
    'evalnet_layer_V':3,
    'evalnet_V':[{'num':30},{'num':30},{'num':30}],
    'targetnet_layer_A':3,
    'targetnet_A':[{'num':30},{'num':30},{'num':30}],
    'targetnet_layer_V':3,
    'targetnet_V':[{'num':30},{'num':30},{'num':30}],
    
}
env

In [48]:
model=DQN(params)

In [49]:
model._build_net()